In [16]:
from transformers import ViTImageProcessor, ViTModel, ViTConfig
from transformers.utils import cached_file
import torch
from PIL import Image
import requests
import timm
import lightning as L

import sys
sys.path.append("../../../../donut/src/test/")
from common.config import cfg

from image_recog_datamodule import ImageClsDataModule
from image_recog_datasets import ImageClsDataset
from class_weighting import compute_class_weights
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import MLFlowLogger
from datetime import datetime

In [17]:
# enable GPU
if torch.backends.mps.is_available():
    print("✅ MPS (Metal Performance Shaders) is available!")
    device = torch.device("mps") 
else:
    print("MPS is not available. Using CPU.")
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")

x = torch.rand(3, 3).to(device)
print(f"Tensor on device: {x.device}")

✅ MPS (Metal Performance Shaders) is available!
PyTorch version: 2.2.0
Tensor on device: mps:0


In [18]:
# load pre defined parameters
local_config_path = cfg.vit_local_config_path

# label encoder
label_encoder = {
    "dot": cfg.dot,
    "scatter": cfg.scatter,
    "horizontal_bar": cfg.horizontal_bar,
    "line": cfg.line,
    "vertical_bar": cfg.vertical_bar,
}

# label decoder
label_decoder = {v: k for k, v in label_encoder.items()}

id2label = {idx: label for idx, label in label_decoder.items()}
label2id = {label: idx for idx, label in label_encoder.items()}

# Dataset parameters
image_cls_height = cfg.vit_image_cls_height
image_cls_width = cfg.vit_image_cls_width

# dataset size multiplier
ds_multiplier = cfg.vit_ds_multiplier

# Datamodule Parameters
initial_augment_prob = cfg.vit_initial_augment_prob
final_augment_prob = cfg.vit_final_augment_prob

# Dataloader parameters
num_workers = cfg.vit_num_workers
batch_size = cfg.vit_batch_size
num_epochs = cfg.vit_num_epochs

# data path
hdf5_file = "../../../data/classificationH5/ImageTransformation.h5"

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

In [19]:
print(f"pretrained model path: {local_config_path}")

print(f"label encoder: {label_encoder}")
print(f"label decoder: {label_decoder}")
print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

pretrained model path: ./pretrained_models/vit-base-patch16-224-in21k
label encoder: {'dot': 0, 'scatter': 1, 'horizontal_bar': 2, 'line': 3, 'vertical_bar': 4}
label decoder: {0: 'dot', 1: 'scatter', 2: 'horizontal_bar', 3: 'line', 4: 'vertical_bar'}
id2label: {0: 'dot', 1: 'scatter', 2: 'horizontal_bar', 3: 'line', 4: 'vertical_bar'}
label2id: {0: 'dot', 1: 'scatter', 2: 'horizontal_bar', 3: 'line', 4: 'vertical_bar'}


In [20]:
# enable MLFLOW
mlflow_logger = MLFlowLogger(
    experiment_name="ViT-image-recognition",
    tracking_uri="http://127.0.0.1:8080", 
)

In [21]:
# load VIT model's config and processor

config = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k",cache_dir=local_config_path)
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k', cache_dir=local_config_path)

In [22]:
type(config)

transformers.models.vit.configuration_vit.ViTConfig

In [23]:
# add customized paramaters into config file

config.num_labels = len(id2label)
config.id2label = id2label
config.label2id = label2id
config.hidden_dropout_prob = 0.1 
config.image_size = 320
config.patch_size = 16
# 320/16=20,
# input token num: 320*320 / 16*16 = 20*20=400 
# add CLS: 400+1=401
# each token seq len: 16*16*3=768

model_name = "google/vit-base-patch16-224-in21k"
config_path = cached_file(model_name, "config.json")

In [24]:
# load pretrained VIT model

model = ViTModel.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    config=config,
    # I want fine tuning the model with Higher Resolution, change the input size from 224 to 320
    # so set ignore mismatched_size right here
    ignore_mismatched_sizes=True,
    cache_dir=local_config_path,
)

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 401, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
print(model)

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation(

In [26]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  ✅ {name}: {param.shape}")
    else:
        print(f"  ❌ {name}: (frozen)")

  ✅ embeddings.cls_token: torch.Size([1, 1, 768])
  ✅ embeddings.position_embeddings: torch.Size([1, 401, 768])
  ✅ embeddings.patch_embeddings.projection.weight: torch.Size([768, 3, 16, 16])
  ✅ embeddings.patch_embeddings.projection.bias: torch.Size([768])
  ✅ encoder.layer.0.attention.attention.query.weight: torch.Size([768, 768])
  ✅ encoder.layer.0.attention.attention.query.bias: torch.Size([768])
  ✅ encoder.layer.0.attention.attention.key.weight: torch.Size([768, 768])
  ✅ encoder.layer.0.attention.attention.key.bias: torch.Size([768])
  ✅ encoder.layer.0.attention.attention.value.weight: torch.Size([768, 768])
  ✅ encoder.layer.0.attention.attention.value.bias: torch.Size([768])
  ✅ encoder.layer.0.attention.output.dense.weight: torch.Size([768, 768])
  ✅ encoder.layer.0.attention.output.dense.bias: torch.Size([768])
  ✅ encoder.layer.0.intermediate.dense.weight: torch.Size([3072, 768])
  ✅ encoder.layer.0.intermediate.dense.bias: torch.Size([3072])
  ✅ encoder.layer.0.output.d

In [27]:
# freeze some layers

# fine tuning embedding layer
for param in model.embeddings.parameters():
    param.requires_grad = True

# freeze the first 6 layers(total 12)
for name, param in model.encoder.named_parameters():
    if any(f"layer.{i}" in name for i in range(6)):  
        param.requires_grad = False

# 🔥 fine tuning the last 6 layers
for name, param in model.encoder.named_parameters():
    if any(f"layer.{i}" in name for i in range(6, 12)):  
        param.requires_grad = True

# 🔥 fine tune pooling layer
for param in model.pooler.parameters():
    param.requires_grad = True

In [28]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable Params: {trainable_params:,} / {total_params:,}")


Trainable Params: 44,018,688 / 86,545,920


In [29]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  ✅ {name}: {param.shape}")
    else:
        print(f"  ❌ {name}: (frozen)")

  ✅ embeddings.cls_token: torch.Size([1, 1, 768])
  ✅ embeddings.position_embeddings: torch.Size([1, 401, 768])
  ✅ embeddings.patch_embeddings.projection.weight: torch.Size([768, 3, 16, 16])
  ✅ embeddings.patch_embeddings.projection.bias: torch.Size([768])
  ❌ encoder.layer.0.attention.attention.query.weight: (frozen)
  ❌ encoder.layer.0.attention.attention.query.bias: (frozen)
  ❌ encoder.layer.0.attention.attention.key.weight: (frozen)
  ❌ encoder.layer.0.attention.attention.key.bias: (frozen)
  ❌ encoder.layer.0.attention.attention.value.weight: (frozen)
  ❌ encoder.layer.0.attention.attention.value.bias: (frozen)
  ❌ encoder.layer.0.attention.output.dense.weight: (frozen)
  ❌ encoder.layer.0.attention.output.dense.bias: (frozen)
  ❌ encoder.layer.0.intermediate.dense.weight: (frozen)
  ❌ encoder.layer.0.intermediate.dense.bias: (frozen)
  ❌ encoder.layer.0.output.dense.weight: (frozen)
  ❌ encoder.layer.0.output.dense.bias: (frozen)
  ❌ encoder.layer.0.layernorm_before.weight: (f

In [30]:
# initilize DataModule

datamodule = ImageClsDataModule(
    hdf5_file=hdf5_file,
    batch_size=batch_size,
    ds_total_len=ds_multiplier,
    initial_augment_prob=initial_augment_prob,
    final_augment_prob=final_augment_prob,
    num_epochs=num_epochs,
    device=device,
    num_workers=num_workers,
    image_height=image_cls_height,
    image_width=image_cls_width
)

In [31]:
from typing import Any, Callable
import torch.nn as nn

from lightning.pytorch.core.optimizer import LightningOptimizer
from torch.optim.optimizer import Optimizer
import torch.nn.functional as F
from lightning.pytorch.callbacks import Callback, ModelCheckpoint, EarlyStopping


class ViTClassifier(L.LightningModule):
    def __init__(self, pretrained_model, config, num_class, lr=None):
        """
        Params:
            pretrained_model: The loaded ViT model
            config:(transformers.models.vit.configuration_vit.ViTConfig) the model's config file
            num_class: the numer of classes
            lr: learning rate
        """
        super().__init__()
        # saving the information in checkpoints and YAML files
        self.save_hyperparameters()

        self.model = pretrained_model
        self.config = config
        self.num_class = num_class
        self.learning_rate = (
            lr if lr is not None else self.hparams.get("learning_rate", 1e-4)
        )

        # define what layers in the pretrained model should be fine-tuning
        # fine tune the embedding (positional embedding) layer
        for param in self.model.embeddings.parameters():
            param.requires_grad = True

        # freeze the first 9 layers(total 12)
        for name, param in self.model.encoder.named_parameters():
            if any(f"layer.{i}" in name for i in range(9)):
                param.requires_grad = False

        # fine tuning the last 3 layers
        for name, param in self.model.encoder.named_parameters():
            if any(f"layer.{i}" in name for i in range(9, 12)):
                param.requires_grad = True

        # fine tune pooling layer
        for param in self.model.pooler.parameters():
            param.requires_grad = True

        # define the classfier head
        self.classifier = nn.Linear(
            in_features=config.hidden_size, out_features=self.num_class
        )

    def forward(self, x):
        outputs = self.model(x)
        # In VIT model, we rely on CLS token to do classification
        cls_token = outputs.last_hidden_state[:, 0, :]
        # classifier head
        logits = self.classifier(cls_token)

        return logits

    def training_step(self, batch, batch_idx):
        images, labels = batch
        images = images.to(self.device)
        labels = labels.to(self.device)

        logits = self(images)
        loss = F.cross_entropy(logits, labels)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        images = images.to(self.device)
        labels = labels.to(self.device)
        logits = self(images)
        val_loss = F.cross_entropy(logits, labels)
        acc = (logits.argmax(dim=1) == labels).float().mean()
        self.log("val_loss", val_loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

        return {"val_loss": val_loss, "val_acc": acc}

    def predict_step(self, batch, batch_idx):
        images, _ = batch
        logits = self(images)
        preds = torch.argmax(logits, dim=1)

        return preds

    def on_train_epoch_end(self):

        train_loss = self.trainer.callback_metrics.get("train_loss")
        if train_loss is not None:
            print(f"Train Epoch {self.current_epoch + 1} - Avg Loss: {train_loss:.4f}")

        # mlflow
        self.logger.log_metrics(
            {
                "manual_record_train_loss": train_loss.item()
            },
            step=self.current_epoch,
        )

    def on_validation_epoch_end(self):

        val_loss = self.trainer.callback_metrics.get("val_loss")
        val_acc = self.trainer.callback_metrics.get("val_acc")
        if val_loss is not None and val_acc is not None:
            print(
                f"Val Epoch {self.current_epoch + 1} - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}"
            )

        # mlflow
        self.logger.log_metrics(
            {
                "manual_record_val_loss": val_loss.item(),
                "manual_record_val_acc": val_acc.item(),
            },
            step=self.current_epoch,
        )

    def on_fit_start(self):
        print(f"✅ Model is on device: {next(self.parameters()).device}")
        if self.logger:
            self.logger.log_hyperparams(self.hparams)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3, # trigger lr scheduler if without imporvement 3 times
            verbose=True,
        )

        # decrease lr based off val loss automatically
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1, # do step each 1 epoch
            },
        }

In [32]:
VitModel=ViTClassifier(pretrained_model=model.to(device),config=config,num_class=len(id2label))

/opt/homebrew/Caskroom/miniforge/base/envs/torch/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'pretrained_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pretrained_model'])`.


In [ ]:
class ViTCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        lr = trainer.optimizers[0].param_groups[0]['lr']
        print(f"Learning Rate after Epoch {trainer.current_epoch + 1}: {lr:.6f}")


# checkpoint
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    dirpath=f"ViTCheckpoints/{timestamp}/", 
    filename="vit-best-{epoch:02d}-{val_acc:.4f}",
    save_weights_only=True,
    verbose=True,
)

# early stopping
early_stop_callback = EarlyStopping(
    monitor="val_acc",
    patience=5, # stop the training if without improvement 5 epochs continusly
    mode="max",
    verbose=True,
)

In [34]:
# need to use self.save_hyperparameters() first
# mlflow_logger.log_hyperparams(model.hparams)

In [ ]:
# # fast test

# trainer = L.Trainer(
#     fast_dev_run=True, 
#     limit_train_batches=3,
#     limit_val_batches=3,
#     logger=mlflow_logger,
#     callbacks=[ViTCallback(), checkpoint_callback, early_stop_callback], 
# )

In [36]:
# trainer.fit(VitModel,datamodule)

In [37]:
trainer = L.Trainer(
    max_epochs=num_epochs,
    callbacks=[ViTCallback(), checkpoint_callback, early_stop_callback],
    logger=mlflow_logger,
    limit_train_batches=30,     
    limit_val_batches=10,
    enable_checkpointing=True,
    log_every_n_steps=1,
    gradient_clip_val=1.0,
    enable_model_summary=True,
    accumulate_grad_batches=4
)


trainer.fit(VitModel,datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name       | Type     | Params | Mode 
------------------------------------------------
0 | model      | ViTModel | 86.5 M | eval 
1 | classifier | Linear   | 3.8 K  | train
------------------------------------------------
44.0 M    Trainable params
42.5 M    Non-trainable params
86.5 M    Total params
346.199   Total estimated model params size (MB)
1         Modules in train mode
227       Modules in eval mode


✅ Model is on device: mps:0


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniforge/base/envs/torch/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:420: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Val Epoch 1 - Loss: 1.6380, Acc: 0.1719


...

Epoch 16, global step 136: 'val_acc' was not in top 1


Val Epoch 17 - Loss: 0.0943, Acc: 0.9812
📉 Learning Rate after Epoch 17: 0.000100
Train Epoch 17 - Avg Loss: 0.1485


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_acc did not improve in the last 5 records. Best score: 0.991. Signaling Trainer to stop.
Epoch 17, global step 144: 'val_acc' was not in top 1


Val Epoch 18 - Loss: 0.0849, Acc: 0.9844
📉 Learning Rate after Epoch 18: 0.000100
Train Epoch 18 - Avg Loss: 0.1818
